# **Importing Libraries**

In [1]:
'''
Install dari terminal

pip install --upgrade --force-reinstall litellm crewai langchain-groq duckduckgo-search ddgs apscheduler "pydantic[email]" langchain-community
pip install fastapi-sso
'''

'\nInstall dari terminal\n\npip install --upgrade --force-reinstall litellm crewai langchain-groq duckduckgo-search ddgs apscheduler "pydantic[email]" langchain-community\npip install fastapi-sso\n'

In [2]:
# AI Agent
import os
from dotenv import load_dotenv
from pathlib import Path
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import concurrent.futures
import litellm
import crewai.llm as _crewai_llm

# API
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Form
from fastapi.middleware.cors import CORSMiddleware

C:\Users\F4QIH\AppData\Local\Temp\ipykernel_18056\780510582.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
env_path = Path('backend/.env')

load_dotenv(dotenv_path=env_path, override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError("GROQ_API_KEY belum disetel di environment lokal.")
os.getenv("OPENAI_API_KEY")

'sk_dummy'

# **Fixing Bug Between CrewAI & Groq**

Menghapus cache_breakpoint pada message dari CrewAI yang tidak kompatibel dengan Groq (cache_breakpoint hanya compatibel dengan Anthropic)

In [4]:
def _clean_messages(messages):
    if not messages:
        return messages
    return [
        {k: v for k, v in m.items() if k != "cache_breakpoint"}
        if isinstance(m, dict) else m
        for m in messages
    ]

_orig_completion = litellm.completion
_orig_acompletion = litellm.acompletion

def _patched_completion(*args, **kwargs):
    if "messages" in kwargs:
        kwargs["messages"] = _clean_messages(kwargs["messages"])
    return _orig_completion(*args, **kwargs)

async def _patched_acompletion(*args, **kwargs):
    if "messages" in kwargs:
        kwargs["messages"] = _clean_messages(kwargs["messages"])
    return await _orig_acompletion(*args, **kwargs)

litellm.completion = _patched_completion
litellm.acompletion = _patched_acompletion

# CrewAI kadang mengimpor completion langsung, jadi timpa juga referensinya
for _name in ("completion", "acompletion"):
    if hasattr(_crewai_llm, _name):
        setattr(_crewai_llm, _name,
                _patched_completion if _name == "completion" else _patched_acompletion)

print("Patch cache_breakpoint aktif (level litellm)")

Patch cache_breakpoint aktif (level litellm)


# **LLM API**

In [5]:
MODEL_ID = "groq/openai/gpt-oss-120b"
my_llm = LLM(
    model=MODEL_ID,
    api_key=groq_api_key,
    temperature=1,
    reasoning_effort="low",
    reasoning_format="hidden"
)

# **Tool Web Search**

In [13]:
search_instance = DuckDuckGoSearchRun()

@tool("web_search")
def web_search(query: str):
    """Useful for finding real-time data on the internet.
    Use specific and concise search keywords."""
    try:
        return search_instance.run(query)
    except Exception as e:
        return f"Search failed: {e}. try using a shorter keyword."

# **Defining Agents (Researcher & Writer)**

In [14]:
researcher = Agent(
    role='Lead Tech Researcher',
    goal='Find and summarize 3 main trends about {topic}',
    backstory='''You are a senior researcher at a global technology firm.
    Your expertise lies in finding valid information and transforming it into a bullet-point report.''',
    tools=[web_search],
    llm=my_llm,
    verbose=True,
    max_iter=3
)

In [15]:
writer = Agent(
    role='Tech Content Strategist',
    goal='Drafting educational blog articles based on technical research findings.',
    backstory='''You are an expert technology blog writer.
    Your writing style is friendly, professional, and easy to understand.''',
    llm=my_llm,
    verbose=True
)

# **Tasks**

In [16]:
task_research = Task(
    description='Use the web_search tool to research {topic}. Identify 3 significant innovations.',
    expected_output='Research report containing 3 key points of innovation.',
    agent=researcher
)

task_write = Task(
    description='Write a 3-paragraph blog post based on a research report from a researcher.',
    expected_output='Complete blog articles in Markdown format.',
    agent=writer
)

# **Crew (Orchestrator)**

In [17]:
my_crew = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

# **Run Agentic Workflow**

In [11]:
if __name__ == "__main__":
    print("--- [CrewAI Multi-Agent Execution Started] ---")
    try:
        # Menggunakan kickoff_async() dan await karena berjalan di dalam event loop Jupyter/Colab
        result = await my_crew.kickoff_async(inputs={'topic': 'Agentic AI Workflow'})
        print("\n--- [Hasil Akhir] ---")
        print(result)
    except Exception as e:
        print(f"Terjadi error: {e}")

--- [CrewAI Multi-Agent Execution Started] ---


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.30                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 05fc12c5-6158-43cb-9137-d3b3a1f5713d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Gunakan tool web_search untuk riset Agentic AI Workflow. Identifikasi 3 inovasi penting untuk tahun      │
│  2026.                                                                                                          │
│  ID: 5e8bc2a3-ae93-47ac-94a6-16b697e926fa                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Tech Researcher                                                                                    │
│                                                                                                                 │
│  Task: Gunakan tool web_search untuk riset Agentic AI Workflow. Identifikasi 3 inovasi penting untuk tahun      │
│  2026.                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'Agentic AI workflow trends 2026'}                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (12 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Buat artikel blog 3 paragraf berdasarkan laporan riset dari researcher.                                  │
│  Agent: Tech Content Strategist                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- [Hasil Akhir] ---
## Agentic AI Workflow 2026: Tiga Inovasi yang Mengubah Cara Kita Bekerja dengan AI  

Agentic AI semakin dewasa, bukan lagi sekadar model “pintar” yang menunggu perintah, melainkan agen‑agen yang **mandiri**, **adaptif**, dan **taat aturan**. Riset terbaru mengidentifikasi tiga inovasi utama yang menjadi fondasi workflow AI tahun 2026: *Self‑Optimizing Prompt Chains*, *Dynamic Tool‑Selection Marketplace*, dan *Governance‑by‑Constraint Engine*. Ketiganya bukan sekadar fitur tambahan, melainkan perubahan paradigma yang memberi manfaat signifikan bagi bisnis dan tim teknis.

### 1. Self‑Optimizing Prompt Chains – Prompt yang Belajar Sendiri  
Agen‑AI kini dapat **menghasilkan, menguji, dan memperbaiki rangkaian prompt secara otomatis**. Dengan memanfaatkan meta‑learning, agen memantau metrik kualitas seperti akurasi, latency, dan biaya, lalu melakukan pencarian gradient‑free (mis. Bayesian optimisation) untuk menyesuaikan struktur prompt secara real‑time. Hasilnya?

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 05fc12c5-6158-43cb-9137-d3b3a1f5713d                                                                       │
│  Final Output: ## Agentic AI Workflow 2026: Tiga Inovasi yang Mengubah Cara Kita Bekerja dengan AI              │
│                                                                                                                 │
│  Agentic AI semakin dewasa, bukan lagi sekadar model “pintar” yang menunggu perintah, melainkan agen‑agen yang  │
│  **mandiri**, **adaptif**, dan **taat aturan**. Riset terbaru mengidentifikasi tiga inovasi utama yang menjadi  │
│  fondasi workflow AI tahun 2026: *Self‑Optimizing Prompt Chains*, *Dynamic Tool‑Selection Marketplace*, dan     │
│  *Governance‑by‑Constraint Engine*. Ketiganya bukan sekadar fitur tambahan, melainkan perubahan paradigma yang  │
│  memberi manfaat signifikan bagi bisnis dan tim teknis.                                                         │
│                                                                                                                 │
│  ### 1. Self‑Optimizing Prompt Chains – Prompt yang Belajar Sendiri                                             │
│  Agen‑AI kini dapat **menghasilkan, menguji, dan memperbaiki rangkaian prompt secara otomatis**. Dengan         │
│  memanfaatkan meta‑learning, agen memantau metrik kualitas seperti akurasi, latency, dan biaya, lalu melakukan  │
│  pencarian gradient‑free (mis. Bayesian optimisation) untuk menyesuaikan struktur prompt secara real‑time.      │
│  Hasilnya? Engineer tidak lagi harus menghabiskan berjam‑jam men‑tune prompt secara manual—waktu pengembangan   │
│  **berkurang 30‑50 %**. Selain mempercepat time‑to‑market produk yang menggabungkan banyak model (LLM, vision,  │
│  reasoning), chain yang self‑optimizing juga menjaga konsistensi output ketika data feed berubah secara         │
│  dinamis, menjadikannya ideal untuk aplikasi real‑time seperti monitoring sensor atau analitik streaming.       │
│                                                                                                                 │
│  ### 2. Dynamic Tool‑Selection Marketplace – Pilih Alat Sesuai Kebutuhan                                        │
│  Platform marketplace terintegrasi memungkinkan agen‑AI **memilih, meng‑on‑board, atau men‑swap tools** (API,   │
│  database, robotik, SDK) secara otomatis. Setiap pemanggilan tool diperlakukan sebagai aksi dalam kerangka      │
│  reinforcement learning, sehingga agen menilai SLA (latency, biaya, keandalan) dan memilih tool yang paling     │
│  efisien pada saat itu. Dampaknya sangat nyata: agen dapat **scale ke ribuan tool tanpa mengubah kode**,        │
│  mengoptimalkan biaya operasional dengan beralih antara model LLM‑lite untuk tugas sederhana dan model premium  │
│  ketika akurasi tinggi diperlukan, serta membuka kolaborasi lintas‑vendor (AWS, Azure, GCP, open‑source) tanpa  │
│  risiko lock‑in.                                                                                                │
│                                                                                                                 │
│  ### 3. Governance‑by‑Constraint Engine – Kepatuhan Otomatis Tanpa Mengorbankan Kecepatan                       │
│  Keamanan, privasi, dan regulasi kini dapat dijaga oleh **engine kontrol berbasis constraint programming**      │
│  yang terintegrasi dalam loop agen. Setiap aksi—mis. mengakses data pribadi atau memicu transaksi—harus         │
│  melewati *policy‑check* deklaratif (GDPR, ISO‑27001, 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# **API SETUP**

In [18]:
# Terapkan nest_asyncio di awal
nest_asyncio.apply()

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], 
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {"status": "ok", "service": "colab"}

@app.post("/research_agent")
async def research_agent(message: str = Form(...)):
    result = await my_crew.kickoff_async(inputs={'topic': message}) 
    return {"response": result.raw} 

# --- CARA BENAR MENJALANKAN UVICORN DI JUPYTER ---
config = uvicorn.Config(app, host="0.0.0.0", port=8080)
server = uvicorn.Server(config)

# Gunakan await agar server menumpang pada event loop Jupyter yang sedang berjalan
await server.serve()

INFO:     Started server process [18056]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.30                                                                                       │
│  Latest version:  1.15.34                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 434301f8-759c-4d6d-9593-f292cdb9a214                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the web_search tool to research agentic ai research oppurtunity. Identify 3 significant              │
│  innovations.                                                                                                   │
│  ID: 2884a17d-04a5-47d9-b17b-354585b008a3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Tech Researcher                                                                                    │
│                                                                                                                 │
│  Task: Use the web_search tool to research agentic ai research oppurtunity. Identify 3 significant              │
│  innovations.                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#7) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'agentic AI research opportunities trends 2024'}                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (34 outputs left out)

INFO:     127.0.0.1:49671 - "POST /research_agent HTTP/1.1" 200 OK


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 434301f8-759c-4d6d-9593-f292cdb9a214                                                                       │
│  Final Output: ## The 2026 LLM Landscape: Sparse Activations, Multimodal Reasoning, and Verified Safety         │
│                                                                                                                 │
│  *By Tech Content Strategist – November 2026*                                                                   │
│                                                                                                                 │
│  Large‑language models are hitting a pivotal moment. After years of scaling up dense transformers, researchers  │
│  have converged on three breakthroughs that promise to make AI **cheaper**, **more capable**, and               │
│  **trustworthy** for every user—from cloud‑based enterprises to the smartphone in your pocket.                  │
│                                                                                                                 │
│  ### 1. Hyper‑Efficient “Sparse‑Activating” Architectures                                                       │
│  Imagine a brain that only lights up the neurons it really needs for a particular thought. That’s the essence   │
│  of **sparse‑activating LLMs**: a tiny slice (often < 5 %) of a trillion‑parameter network is engaged for each  │
│  token, thanks to learned routing and mixture‑of‑experts (MoE) mechanisms refined over 2024‑2025. Papers like   │
│  *Dynamic Sparse Transformers* (NeurIPS 2025) showed an 85 % reduction in activation cost while preserving      │
│  state‑of‑the‑art reasoning scores, and Google DeepMind’s **GLaM‑2** proved that a 1.2 T‑parameter MoE can      │
│  match GPT‑4 performance at one‑third the inference cost. The open‑source **SparseMind** project (2026) now     │
│  lets anyone spin up a custom sparse model under an Apache 2.0 license, democratizing what used to be a         │
│  data‑center‑only capability. The payoff is immediate: inference FLOPs drop 10‑20×, cloud‑GPU bills shrink,     │
│  and real‑time AI becomes feasible on edge devices such as smartphones, AR glasses, and even low‑power IoT      │
│  sensors. In short, sparse‑activating models are the de‑facto path to “AI at scale for everyone.”               │
│                                                                                                                 │
│  ### 2. Unified Multi‑Modal Foundations (MM‑LLMs)                                                               │
│  The next frontier is no longer “text‑only” AI but **perception‑plus‑language** systems that ingest text,       │
│  images, video, audio, and structured data within a single transformer backbone. Recent                         │
│  breakthroughs—*Modality‑Alignment Transformers* (MAT) with dual‑attention blocks and *Temporal‑Fusion Tokens*  │
│  that capture video flow—have enabled models to reason across modalities seamlessly. Large self‑supervised      │
│  pre‑training on datasets like **Web4All** (10 B image‑text‑audio triples) aligns these diverse signals at      │
│  scale. Flagship models such as **Meta’s LLaVA‑3** (13 B parameters, >90 % on the new MM‑Bench) and             │
│  **Anthropic’s Claude‑Multimodal‑2** (which can generate SQL on the fly from a screenshot of a table)           │
│  demonstrate that a single model can replace separate OCR, speech‑to‑text, and vision pipelines. Open‑source    │
│  **Mosaic‑MM** (2026) adds plug‑and‑play adapters for 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [18056]
